In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TEMP VIEW order_items_typed AS
SELECT
  order_id
  , TRY_CAST(order_item_id AS INTEGER) AS order_item_id
  , product_id
  , seller_id
  , TRY_CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_timestamp
  , TRY_CAST(price AS DECIMAL(10,2)) AS order_line_price
  , TRY_CAST(freight_value AS DECIMAL(10,2)) as order_line_freight_value
  , _ingested_at
FROM workspace.bronze.order_items
QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id, order_item_id ORDER BY _ingested_at DESC) = 1;

CREATE OR REPLACE TABLE workspace.silver.order_items AS
SELECT * FROM order_items_typed
WHERE order_id IS NOT NULL AND order_item_id IS NOT NULL AND product_id IS NOT NULL AND order_line_price >= 0 AND order_line_price is not null;

CREATE OR REPLACE TABLE workspace.silver.order_items_quarantine AS
SELECT 
* 
, CASE
    WHEN order_id IS NULL OR order_item_id IS NULL OR product_id IS NULL OR seller_id IS NULL
      THEN 'llave nula'
    WHEN order_line_price IS NULL OR order_line_price < 0
      THEN 'precio nulo o negativo'
    WHEN order_line_freight_value IS NULL OR order_line_freight_value < 0
      THEN 'flete nulo o negativo'
  END AS motivo
FROM order_items_typed 
WHERE 1=1
AND (
      order_id IS NULL 
      OR order_item_id IS NULL 
      OR product_id IS NULL
      OR seller_id IS NULL
      OR order_line_price < 0 
      OR order_line_price IS NULL
      OR order_line_freight_value IS NULL
      OR order_line_freight_value < 0
);